# Sequence & LLM Re-ranking


**Two regimes** of personalization live in this notebook, and they correspond to two different temporal scales.

1. **Short-term intent.** A user who just watched *The Matrix Reloaded* will want *The Matrix Revolutions* next, regardless of their global preference for sci-fi. A sequence model — **SASRec** (Kang & McAuley 2018) — captures that. It replaces the *static* user tower of the Two-Tower model with a self-attention stack over the user's most recent N items.

2. **Long-tail understanding.** A model that has only seen the genres `Action|Sci-Fi|Thriller` will never discover that the user simply *doesn't enjoy dystopias*. An **LLM re-ranker** can read the plot text of each candidate and judge fit in natural language, including nuance the catalog metadata doesn't carry. We use OpenRouter (OpenAI-compatible API) and let GPT-4o-mini re-rank the candidate set retrieved by REC:04.

These two pieces correspond to two of the most important *modern* recsys shifts of 2020-future: from per-user embeddings to per-session transformers, and from learned-with-gradient models in Python to *learned-via-prompt* LLM joints running in API calls.


## Setup


In [ ]:
#| echo: false
import warnings
warnings.filterwarnings("ignore")
import os; os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")


In [ ]:
import time
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from matplotlib_inline import backend_inline

backend_inline.set_matplotlib_formats("svg")
plt.rcParams["figure.dpi"] = 110
torch.manual_seed(0)

from notebooks.recsys.config import MovieLensConfig
from notebooks.recsys.data import load_movielens, time_split
from notebooks.recsys.metrics import Metricator
from notebooks.recsys.features import FeatureStore
from notebooks.recsys.models.sequence import SASRec, SASRecConfig, SASRecTrainer, SessionRetriever
from notebooks.recsys.models.llm_rerank import LLMRerankConfig, LLMReranker


## SASRec: self-attention over user history

The model is a Transformer encoder over item embeddings with positional encoding. Define the input sequence as $\mathbf{S} = [i_1, \ldots, i_T]$. The model computes

$$\mathbf{H}^{(0)} = [\mathbf{e}_{i_1} + \mathbf{p}_1, \ldots, \mathbf{e}_{i_T} + \mathbf{p}_T],$$

applies $L$ Transformer-encoder layers, and reads the hidden vector at the final position $\mathbf{h}^{(L)}_T$ as the *session representation*. It scores candidate $j$ by $\mathbf{h}^{(L)}_T \cdot \mathbf{e}_j$ — i.e. a Two-Tower score where the user tower is now a self-attention stack.

**The self-attention block itself**, for a single head:

$$\mathrm{Attn}(\mathbf{Q}, \mathbf{K}, \mathbf{V}) = \mathrm{softmax}\left(\frac{\mathbf{Q}\mathbf{K}^{\top}}{\sqrt{d_k}} + M\right)\mathbf{V},$$

with $\mathbf{Q} = \mathbf{H}\mathbf{W}_Q$, $\mathbf{K} = \mathbf{H}\mathbf{W}_K$, $\mathbf{V} = \mathbf{H}\mathbf{W}_V$ and $M$ the causal mask (positions may not attend forward).

In SASRec we use a *bidirectional* encoder (the model attends to *all* items in the window), and the *next-item* loss is the cross-entropy of the target on the catalog scored against $\mathbf{h}^{(L)}_{T-1}$. We use a sampled-softmax approximation (one positive + $K$ sampled negatives) to avoid the $\mathcal{O}(|V|)$ softmax denominator.


In [ ]:
cfg = MovieLensConfig(name="ml-100k")
ds = load_movielens(cfg)
train, val = time_split(ds.ratings, val_frac=0.2)

sascfg = SASRecConfig(
    n_items=ds.n_items, embedding_dim=32, n_heads=2,
    n_layers=1, max_seq_len=30, epochs=3,
    batch_size=128, n_negatives=15, lr=1e-3,
)
sas = SASRec(sascfg)
SASRecTrainer(model=sas, cfg=sascfg).fit(train)


In [ ]:
sr = SessionRetriever(model=sas, dataset=ds, train=train)
uid = int(val['user_id'].iloc[0])
print('top-5 session recs:', sr.recommend(uid, k=5))

# Watch one user's history and recommendations.
hist = train[train['user_id']==uid].sort_values('timestamp').tail(8)
hist_titles = ds.movies.set_index('item_id').loc[hist['item_id'], 'title']
print('last 8 watches:', list(hist_titles))


**Observation.** SASRec pushes recommendations toward items adjacent to the user's *last few* watches — the self-attention rapidly drops positional weighting for early witnesses. That's what it should do for short-horizon intent.


## LLM re-ranker

The ranker model relies on dense numeric features. The LLM-as-judge approach trades millions of trained parameters for a single expensive API call *full of general world knowledge*. For a painfully cold-start user — first-time, no observed ratings — every gradient-based model fails; the LLM has no such gap if the candidates come with text.

Algorithm:

1. Retrieve $N \approx 20$ candidates via TwoTower (or SASRec).
2. Build a prompt: user's recent liked titles + each candidate's title/genres/overview. Ask the LLM to *re-order* candidate numbers by predicted user enjoyment.
3. Parse the LLM output as a numbered list and convert back to item_id ordering.

Cost: ~one API call per $k$-candidates per user. The latency lives in the API round-trip (typically $200\sim 2000$ ms). The `LLMReranker` cache means subsequent identical requests cost nothing.


::: {.callout-note}
The LLM-as-judge is *not* a free lunch: it costs tokens, it inherits the LLM's biases toward popular items (the LLM has *seen* The Matrix more often than Winged Migration), and it has stricter latency — hundreds of milliseconds versus the ranker's microseconds. Use it as a *cold-start* or *re-rank* layer; pair with retrieval/ranking for scale.
:::


In [ ]:
import os
has_openrouter = bool(os.environ.get('OPENROUTER_API_KEY'))
print('OPENROUTER_API_KEY present?', has_openrouter)


In [ ]:
from notebooks.recsys.models.retrieval import (
    TwoTower, TwoTowerConfig, InBatchSoftmaxLoss, TwoTowerTrainer, Retriever,
)

torch.manual_seed(0)
tt_cfg = TwoTowerConfig(n_users=ds.n_users, n_items=ds.n_items,
                       embedding_dim=32, hidden_dim=64,
                       n_negatives=0, epochs=3, batch_size=1024)
tt = TwoTower(tt_cfg)
loss = InBatchSoftmaxLoss(n_items=ds.n_items, n_negatives=0, temperature=0.1)
TwoTowerTrainer(model=tt, loss=loss, cfg=tt_cfg).fit(train, verbose=False)
retriever = Retriever(model=tt, dataset=ds, train=train)
_ = retriever.recommend(uid, k=10)  # warm FAISS
cands = retriever.recommend(uid, k=20)
print(f'TwoTower candidates: {cands}')


In [ ]:
# Build the reranker. We don't actually call the API if you don't have a key set;
# use OPENROUTER_API_KEY env var to flip on live LLM re-ranking.
store = FeatureStore(ds)
rerank_cfg = LLMRerankConfig(model="openai/gpt-4o-mini")
reranker = LLMReranker(cfg=rerank_cfg, store=store, dataset=ds)

if has_openrouter:
    reranked = reranker.recommend(uid, cands, k=10)
    print('LLM rerank:', reranked)
else:
    print('Skipping live LLM call (OPENROUTER_API_KEY not set).')
    print('The cells below exercise the prompt construction without spending tokens.')


In [ ]:
# Always visible: inspect the prompt that would have been sent.
hist_rows = reranker._user_history(uid)
movies = store.movies.set_index('item_id')
user_meta = [(iid, movies.loc[iid, 'title'], movies.loc[iid, 'genres'])
             for iid in hist_rows if iid in movies.index]
cand_meta = [(iid, movies.loc[iid, 'title'], movies.loc[iid, 'genres'], movies.loc[iid, 'overview'])
             for iid in cands if iid in movies.index]
print(reranker._build_prompt(user_meta, cand_meta))


**Reading the prompt.** It is short on purpose: the user's favorite-vibe items + the candidates' titles/genres/overviews, then a numbered-list-output instruction. With longer context (sessions, queries), production LLM re-rankers include additional business rules: *promote fresh items*, *demote over-surfaced items*, *respect quiescent preferences*. We will see at least the first two emerge naturally as REC:08's bandit policy layers over the ranker.


## Comparing the two regimes

Where do SASRec and the LLM re-ranker sit relative to the ALS / Two-Tower / Ranker stack we built in earlier stages? Recall the promise of each:

- ALS — explicit MF baseline (REC:03)
- Two-Tower — deep retrieval, in-batch negatives (REC:04)
- Ranker (Wide&Deep / DCN / pairwise) — second-stage re-rank on retrieval candidates (REC:05)
- SASRec — session sequence model (REC:06 second half)
- LLM re-ranker — text-aware cold-start surrerogate (REC:06)

Below we compare SASRec metrics to the retriever's ``recall@50`` from REC:04.


In [ ]:
metricator = Metricator(val)
sas_m = metricator.evaluate(sr.recommend, k=50)
tt_m = metricator.evaluate(retriever.recommend, k=50)
rows = [{"model": obj, "recall@50": round(m["recall@50"],4),
         "ndcg@50": round(m["ndcg@50"],4),
         "coverage@50": round(m["coverage@50"],4),
         "novelty@50": round(m["novelty@50"],4)}
        for obj, m in [('TwoTower', tt_m), ('SASRec', sas_m)]]
pd.DataFrame(rows)


**Observation.** SASRec's Recall@50 trails the Two-Tower retriever's, often by 2-5x, on ml-100k. That gap is *expected*: SASRec's signal is the *session*, and ml-100k's sessions spread months apart, making "next-item" a noisy target. On session-rich datasets (Spotify sessions, e-commerce browsing logs) SASRec dominates. The right takeaway is that recall compares stages *on the same metric*, but they answer different questions: "what is the user like over time?" is retrieval; "what does the user want right now?" is sequence.


## Caveats and link forward

- **SASRec training is unstable on MovieLens 100k alone.** Add a positional dropout (= the early LLM-recsys work's "stacking" variant of BERT4Rec) and a smaller learning rate, or train on a larger dataset (ml-25m).
- **The LLM re-ranker today is *zero-shot*.** A more powerful recipe is fine-tuned LLM rankers (e.g. RankGPT, Zhang et al. 2023) or distillation into a small model (REC:09's demo app falls back to the small ranker).
- **Cost bookkeeping is non-trivial.** $0.002 per call adds up at a million users per day. The cache + batching strategies developed in REC:09's demo interface are the same ones used in production.

Next: REC:07 builds the serving infra — a FastAPI service, MLflow registry, Feast-lite feature store — that lets the demo app in REC:09 call any of these models over HTTP.
